## Previous Findings
* We got total of 4548 rows and 7 columns.
* No missing values in the dataset.
* The address column has \n (next line operator) in the address data.
* The price data needs to be converted into fixed e.g. 2nd or 3rd deciaml places.
* The house ages is well distributed between 2.6-9.5 years, where the houses are not so old, and not so new.
* Based on the Area Income, we can say that the properties lie in high income areas(75% people earn more than 60k).
* Based on the room no.s VS bedroom no.s, we can say that the properties are of Villa or Bungalow types.
* The population density is very low, which means the properties are located in semi-rural areas.
* We found that, column index 2 and 3 are needed to be converted into int data type, since no. of rooms cannot be in fractions.

# Imports

In [1]:
import pandas as pd

# Data Imports

In [2]:
df = pd.read_csv("House_Price.csv")

In [3]:
df.sample(n=5)

,Avg. Area Income,House Age,Number of Rooms,Number of Bedrooms,Area Population,Price,Address
1902,76275.33255,5.185436,6.972646,2.22,40354.70525,1.464929e+06,"021 Katherine Ranch Apt. 263\nMurphytown, OH 5..."
3043,46646.71054,6.331878,6.726796,3.45,34006.86530,7.102692e+05,"453 Tyler Prairie Apt. 400\nSwansonton, VI 028..."
2962,64836.65398,6.490524,8.271983,6.30,35163.51684,1.296468e+06,"336 Tiffany Via\nChristophertown, CA 97338-2426"
1733,77098.86887,7.757390,6.246759,3.50,46478.52161,1.637650e+06,"586 Lori Rapids\nSouth Melissa, SC 90542"
2082,60110.37928,5.926384,5.626159,4.19,32536.46587,8.744973e+05,"29222 Smith Rest\nMichelletown, MS 57014-3973"


In [4]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 4548 entries, 0 to 4547
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Avg. Area Income    4548 non-null   float64
 1   House Age           4548 non-null   float64
 2   Number of Rooms     4548 non-null   float64
 3   Number of Bedrooms  4548 non-null   float64
 4   Area Population     4548 non-null   float64
 5   Price               4548 non-null   float64
 6   Address             4548 non-null   str    
dtypes: float64(6), str(1)
memory usage: 636.7 KB


## Changing column names

In [5]:
columns = df.columns
new_cols = []
for cols in columns:
    cols = cols.lower()
    cols = cols.replace(" ", "_").replace(".", "")
    new_cols.append(cols)
new_cols

['avg_area_income',
 'house_age',
 'number_of_rooms',
 'number_of_bedrooms',
 'area_population',
 'price',
 'address']

In [6]:
df.columns = new_cols
df.head(5)

,avg_area_income,house_age,number_of_rooms,number_of_bedrooms,area_population,price,address
0,79545.45857,5.682861,7.009188,4.09,23086.80050,1.059034e+06,"208 Michael Ferry Apt. 674\nLaurabury, NE 3701..."
1,79248.64245,6.002900,6.730821,3.09,40173.07217,1.505891e+06,"188 Johnson Views Suite 079\nLake Kathleen, CA..."
2,61287.06718,5.865890,8.512727,5.13,36882.15940,1.058988e+06,"9127 Elizabeth Stravenue\nDanieltown, WI 06482..."
3,63345.24005,7.188236,5.586729,3.26,34310.24283,1.260617e+06,USS Barnett\nFPO AP 44820
4,59982.19723,5.040555,7.839388,4.23,26354.10947,6.309435e+05,USNS Raymond\nFPO AE 09386


# Feature Engineering the `address` column (Phase 3 work, done here for demonstration purposes only)

In [7]:
df["address"]

0       208 Michael Ferry Apt. 674\nLaurabury, NE 3701...
1       188 Johnson Views Suite 079\nLake Kathleen, CA...
2       9127 Elizabeth Stravenue\nDanieltown, WI 06482...
3                               USS Barnett\nFPO AP 44820
4                              USNS Raymond\nFPO AE 09386
                              ...                        
4543          97160 Tracy Junction\nErinborough, WY 73884
4544            0630 Wilson Shoal\nNorth Philip, AK 91611
4545                PSC 2681, Box 5759\nAPO AA 82431-2879
4546    04117 Bennett Greens\nGonzalezfort, NJ 86640-8362
4547                                              55454 M
Name: address, Length: 4548, dtype: str

In [8]:
# 1. Extract Street/Unit (Everything before the newline)
df['street'] = df["address"].str.split('\n').str[0]

# 2. Extract the second line for further parsing
df['_last_line'] = df["address"].str.split('\n').str[1]

# 3. Extract Zip Code (supports 12345 and 12345-6789)
df['zip_code'] = df['_last_line'].str.extract(r'(\d{5}(?:-\d{4})?)$')

# 4. Extract State (Two uppercase letters followed by a space and the zip)
df['state'] = df['_last_line'].str.extract(r'\s([A-Z]{2})\s+\d')

# 5. Extract City
# Regex logic: Capture everything at the start of the line until the state code
df['city'] = df['_last_line'].str.extract(r'^(.*?)(?:,\s[A-Z]{2}|[A-Z]{2})\s+\d')
df['city'] = df['city'].str.replace(',', '').str.strip()

# 6. Feature: Is Military/Diplomatic Address?
# Common indicators: APO (Army Post Office), FPO (Fleet Post Office), DPO (Diplomatic Post Office)
df['is_military'] = df["address"].str.contains(r'APO|FPO|DPO', regex=True).astype("int16")

# Clean up temporary column
df = df.drop(columns=['_last_line','address'])
df.head(20)

,avg_area_income,house_age,number_of_rooms,number_of_bedrooms,area_population,price,street,zip_code,state,city,is_military
0,79545.45857,5.682861,7.009188,4.09,23086.80050,1.059034e+06,208 Michael Ferry Apt. 674,37010-5101,NE,Laurabury,0
1,79248.64245,6.002900,6.730821,3.09,40173.07217,1.505891e+06,188 Johnson Views Suite 079,48958,CA,Lake Kathleen,0
2,61287.06718,5.865890,8.512727,5.13,36882.15940,1.058988e+06,9127 Elizabeth Stravenue,06482-3489,WI,Danieltown,0
3,63345.24005,7.188236,5.586729,3.26,34310.24283,1.260617e+06,USS Barnett,44820,AP,FPO,1
4,59982.19723,5.040555,7.839388,4.23,26354.10947,6.309435e+05,USNS Raymond,09386,AE,FPO,1
5,80175.75416,4.988408,6.104512,4.04,26748.42842,1.068138e+06,06039 Jennifer Islands Apt. 443,16077,KS,Tracyport,0
6,64698.46343,6.025336,8.147760,3.41,60828.24909,1.502056e+06,4759 Daniel Shoals Suite 442,20247,CO,Nguyenburgh,0
7,78394.33928,6.989780,6.620478,2.42,36516.35897,1.573937e+06,972 Joyce Viaduct,17778-6483,TN,Lake William,0
8,59927.66081,5.362126,6.393121,2.30,29387.39600,7.988695e+05,USS Gilbert,20957,AA,FPO,1
9,81885.92718,4.423672,8.167688,6.10,40149.96575,1.545155e+06,Unit 9446 Box 0958,97025,AE,DPO,1


# Null data removal

In [9]:
df = df.dropna()

# Dtype change and compression

In [10]:
df["avg_area_income"] = df["avg_area_income"].round(2)
df["house_age"] = df["house_age"].round(1).astype("float32")
df["number_of_rooms"] = df["number_of_rooms"].astype("int8")
df["number_of_bedrooms"] = df["number_of_bedrooms"].astype("int8")
df["area_population"] = df["area_population"].round(3)
df["price"] = df["price"].round(3)
df["street"] = df["street"].astype("str")
df["zip_code"] = df["zip_code"].astype("str")
df["state"] = df["state"].astype("str")
df["city"] = df["city"].astype("str")

df.head(5)

,avg_area_income,house_age,number_of_rooms,number_of_bedrooms,area_population,price,street,zip_code,state,city,is_military
0,79545.46,5.7,7,4,23086.800,1059033.558,208 Michael Ferry Apt. 674,37010-5101,NE,Laurabury,0
1,79248.64,6.0,6,3,40173.072,1505890.915,188 Johnson Views Suite 079,48958,CA,Lake Kathleen,0
2,61287.07,5.9,8,5,36882.159,1058987.988,9127 Elizabeth Stravenue,06482-3489,WI,Danieltown,0
3,63345.24,7.2,5,3,34310.243,1260616.807,USS Barnett,44820,AP,FPO,1
4,59982.20,5.0,7,4,26354.109,630943.489,USNS Raymond,09386,AE,FPO,1


In [11]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 4547 entries, 0 to 4546
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   avg_area_income     4547 non-null   float64
 1   house_age           4547 non-null   float32
 2   number_of_rooms     4547 non-null   int8   
 3   number_of_bedrooms  4547 non-null   int8   
 4   area_population     4547 non-null   float64
 5   price               4547 non-null   float64
 6   street              4547 non-null   str    
 7   zip_code            4547 non-null   str    
 8   state               4547 non-null   str    
 9   city                4547 non-null   str    
 10  is_military         4547 non-null   int16  
dtypes: float32(1), float64(3), int16(1), int8(2), str(4)
memory usage: 1.2 MB


# Exporting Cleaned Data

In [ ]:
df.to_csv("cleaned_house_price.csv")

# Findings
* The data is clean and ready for further analysis.
* Extracted factors like street, zip_code, state, city ,etc from the `address` column.
* The last column states that the property is either a Government(1) or Private property(0).
* The dtypes were reconfigures based on the data types and value range of each column to optimize memory usage.
* The rows with missing values are dropped.
* The factors like `room_rumbers`, etc which were supposed to be in int logically but found in float format were taken care of.
* House age simplified by rounging into 1th decimal place.
### Thereby the data is cleaned and ready for further usage